In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/dl06/cleaned_data_2.csv')

df.shape
df.head()

,tieu_de,dia_chi,gia_ban,gia_ban_num,dien_tich,dien_tich_num,price_per_m2,so_phong_ngu_num,so_phong_ve_sinh_num,quan,quan_encoded,text_clean
0,🍀VÂN KIỀU AN CƯ🍀 40m2 - Hẻm Ô Tô - Gần mặt tiề...,"Đường Lê Quang Định, Phường 7, Quận Bình Thạnh...","3,85 tỷ",3.850000e+09,36 m²,36.0,1.069444e+08,2.0,2.0,binh_thanh,0,vân kiều an_cư 40m2 hẻm ô_tô mặt_tiền ngang to...
1,"9.xty,Nhà Hẻm Xe Hơi,Xây Mới Khu VIP giáp Phạm...","Đường Phạm Văn Đồng, Phường 13, Quận Bình Thạn...","9,79 tỷ",9.790000e+09,62 m²,62.0,1.579032e+08,4.0,3.0,binh_thanh,0,9 xty hẻm xe_hơi xây khu vip giáp phạm văn đồn...
2,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,"Đường Nơ Trang Long, Phường 13, Quận Bình Thạn...","7,2 tỷ",7.200000e+09,54 m²,54.0,1.333333e+08,3.0,2.0,binh_thanh,0,xe_hơi ngủ ngang 7m hiếm nơ trang long p13 bt ...
3,🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TIỆN XÂY M...,"Đường Xô Viết Nghệ Tĩnh, Phường 21, Quận Bình ...","8,5 tỷ",8.500000e+09,83 m²,83.0,1.024096e+08,3.0,3.0,binh_thanh,0,hẻm ôtô hàng xanh tiện xây 8 5 tỷ e45 hẻm ôtô ...
4,Nhà hẻm cách đường mặt tiền 20m,"Đường Chu Văn An, Phường 12, Quận Bình Thạnh, ...","2,85 tỷ",2.850000e+09,18 m²,18.0,1.583333e+08,2.0,1.0,binh_thanh,0,hẻm đường mặt_tiền 20m hẻm đường mặt_tiền 20m ...


PHẦN 1 — TF-IDF + COSINE (SKLEARN)

In [4]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

tfidf_matrix = tfidf.fit_transform(df["text_clean"])

In [5]:
#COSINE SIMILARITY
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [6]:
def recommend_cosine(idx, top_n=5):
    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]

    result = df.iloc[indices][[
        "tieu_de",
        "gia_ban",
        "dien_tich",
        "quan"
    ]].copy()

    result["similarity_score"] = scores

    return result

PHẦN 2 — GENSIM

In [7]:
#TOKENIZE LIST
texts = [text.split() for text in df["text_clean"]]

In [8]:
!pip install gensim pyvi wordcloud

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.8 MB/s eta 0:00:00


In [9]:
#BUILD MODEL
from gensim import corpora
from gensim.models import TfidfModel
from gensim.similarities import MatrixSimilarity

dictionary = corpora.Dictionary(texts)

corpus = [dictionary.doc2bow(text) for text in texts]

tfidf_model = TfidfModel(corpus)

corpus_tfidf = tfidf_model[corpus]

index = MatrixSimilarity(corpus_tfidf)

In [10]:
#HÀM RECOMMEND GENSIM
def recommend_gensim(idx, top_n=5):
    vec = corpus_tfidf[idx]
    sims = list(enumerate(index[vec]))

    sims = sorted(sims, key=lambda x: x[1], reverse=True)
    sims = sims[1:top_n+1]

    indices = [i[0] for i in sims]
    scores = [i[1] for i in sims]

    result = df.iloc[indices][[
        "tieu_de",
        "gia_ban",
        "dien_tich",
        "quan"
    ]].copy()

    result["similarity_score"] = scores
    return result

#PHẦN 3 — HYBRID RECOMMENDER

In [11]:
#CONTENT SIM (reuse cosine)
content_sim = cosine_sim

In [12]:
#PRICE SIMILARITY
import numpy as np

price = df["gia_ban_num"].values

# normalize
price_norm = (price - price.min()) / (price.max() - price.min())

price_sim = 1 - np.abs(price_norm[:, None] - price_norm[None, :])

In [13]:
#LOCATION SIMILARITY
location = df["quan_encoded"].values

location_sim = (location[:, None] == location[None, :]).astype(int)

In [14]:
#HYBRID FORMULA
alpha = 0.5   # content
beta = 0.25   # price
gamma = 0.25  # location

hybrid_sim = (
    alpha * content_sim +
    beta * price_sim +
    gamma * location_sim
)

In [15]:
#HÀM RECOMMEND HYBRID
def recommend_hybrid(idx, top_n=5):
    sim_scores = list(enumerate(hybrid_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    indices = [i[0] for i in sim_scores]

    result = df.iloc[indices][[
        "tieu_de",
        "gia_ban",
        "dien_tich",
        "quan"
    ]].copy()

    result["similarity_score"] = [round(sim_scores[i][1], 3) for i in range(len(sim_scores))]

    return result

In [16]:
import joblib



# 👉 dataframe gốc (PHẢI GIỐNG df bạn dùng trong hàm)
joblib.dump(df, "/content/drive/MyDrive/dl06/df_recommend.pkl")

['/content/drive/MyDrive/dl06/df_recommend.pkl']

In [17]:
# 👉 ma trận similarity
joblib.dump(hybrid_sim, "/content/drive/MyDrive/dl06/hybrid_sim.pkl")


['/content/drive/MyDrive/dl06/hybrid_sim.pkl']

## TEST

In [ ]:
print(df.loc[0, "tieu_de"])

In [ ]:
df.iloc[0][[
    "tieu_de",
    "gia_ban",
    "dien_tich",
    "quan"
]]

In [ ]:
recommend_cosine(0)

In [ ]:
recommend_gensim(0)

In [ ]:
recommend_hybrid(0)